In [1]:
!pip install ultralytics easyocr opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 33.0 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
import re
from collections import defaultdict, deque

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/anpr_detector.pt")
reder = easyocr.Reader(['en'],gpu=True)
plate_pattern = re.compile(r"^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [5]:
def correct_plate_format(ocr_text):
    mapping_num_to_alpha = {"0":"O","1":"I","5":"S","8":"B"}
    mapping_num_to_alpha = {"O":"0","I":"1","Z":"2","S":"5","B":"8"}

    ocr_text= ocr_text.upper().replace(" ", "")
    if len(ocr_text) != 7:
      return "" #removing if all numbers are not read
    corrected =[]
    for i, ch in enumerate(ocr_text):
      if i< 2 or i>=4:
        if ch.isdigit() and ch in mapping_num_to_alpha :
          corrected.append(mapping_num_to_alpha[ch])
        elif ch.isalpha():
          corrected.append(ch)
        else:
          return ""
      else:
        if ch.isalpha() and ch in mapping_alpha_to_num :
          corrected.append(mapping_alpha_to_num[ch])
        elif ch.isalpha():
          corrected.append(ch)
        else:
          return ""
    return "".join(corrected)





In [6]:
def recognize_plate(plate_crop):
    if plate_crop.size == 0:
        return ""

    # Preprocess for OCR
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2
          .THRESH_OTSU)
    plate_resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation
          =cv2.INTER_CUBIC)

    try:
        ocr_result = reader.readtext(
            plate_resized, detail=0, allowlist
                = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
        )
        if len(ocr_result) > 0:
            candidate = correct_plate_format(ocr_result[0])
            if candidate and plate_pattern.match(candidate):
                return candidate
    except:
        pass

    return ""

In [7]:
input_video = "vehicle video.mp4"
output_video = "output with licensev3.mp4"

cap = cv2.VideoCapture(input_video)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc,
                      cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(3)), int(cap.get(4))))

CONF_THRESH = 0.3

In [8]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    for r in results:
        boxes = r.boxes
        for box in boxes:
            conf = float(box.conf.cpu().numpy())
            if conf < CONF_THRESH:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            plate_crop = frame[y1:y2, x1:x2]

            # OCR with correction
            text = recognize_plate(plate_crop)

            # Stabilize text using history
            box_id = get_box_id(x1, y1, x2, y2)
            stable_text = get_stable_plate(box_id, text)

            # Draw simple rectangle around license plate
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 3)

            # Overlay zoomed-in plate above detected plate
            if plate_crop.size > 0:
                overlay_h, overlay_w = 150, 400
                plate_resized = cv2.resize(plate_crop, (overlay_w, overlay_h))

                oy1 = max(0, y1 - overlay_h - 40)
                ox1 = x1
                oy2, ox2 = oy1 + overlay_h, ox1 + overlay_w

                if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
                    frame[oy1:oy2, ox1:ox2] = plate_resized

                # Show stabilized OCR text above overlay
                if stable_text:
                    cv2.putText(frame, stable_text, (ox1, oy1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 2, (0,0,0), 6)  # black outline
                    cv2.putText(frame, stable_text, (ox1, oy1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 3)  # white text

    out.write(frame)
    cv2.imshow("Annotated Video", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [9]:
cap.release()
out.release()
cv2.destroyAllWindows()
print("annotated video saved as ",output_video)

annotated video saved as  output with licensev3.mp4
